# RAG Pipeline Test

Test the full pipeline: indexes, retrievers, prompts, and end-to-end answer generation.

In [1]:
import json
import sys
sys.path.append('..')

import config
from rag.build_index import load_documents, build_minsearch_index, build_vector_index, get_embedder
from rag.search import search_keyword, search_vector, search_hybrid, rerank
from rag.rag_pipeline import ask

In [2]:
# --- Load indexes ---
docs = load_documents()
print(f"Documents: {len(docs)}")

kw_index = build_minsearch_index(docs)
embeddings = build_vector_index(docs)
embedder = get_embedder()
print(f"Embeddings shape: {embeddings.shape}")

Documents: 489


ValueError: too many values to unpack (expected 2)

In [ ]:
search_keyword(kw_index, "How much stamp duty for a RM500k property?", k=5)

[{'id': 'bec65253-92ab-4fa6-91fc-6ce1b2d374ba',
  'source': 'iproperty',
  'category': 'legal_cost',
  'title': 'Property Stamp Duty in Malaysia: How to Calculate',
  'url': 'https://www.iproperty.com.my/guides/stamp-duty-spa-legal-fees-owning-a-house-malaysia-24760',
  'text': 'Property Stamp Duty in Malaysia: How to Calculate Stamp Duty Malaysia in 2026?',
  'score': 0.0},
 {'id': '7bc86587-b76a-4895-8a58-0aaaa9a87929',
  'source': 'iproperty',
  'category': 'legal_cost',
  'title': 'Property Stamp Duty in Malaysia: How to Calculate',
  'url': 'https://www.iproperty.com.my/guides/stamp-duty-spa-legal-fees-owning-a-house-malaysia-24760',
  'text': 'What Is Property Stamp Duty? 2.',
  'score': 0.0},
 {'id': '5e1aed3c-25ff-4473-9f6f-bf0e411b2507',
  'source': 'iproperty',
  'category': 'legal_cost',
  'title': 'Property Stamp Duty in Malaysia: How to Calculate',
  'url': 'https://www.iproperty.com.my/guides/stamp-duty-spa-legal-fees-owning-a-house-malaysia-24760',
  'text': 'What Is Pro

## Test Questions

In [ ]:
test_questions = [
    "How much stamp duty for a RM500k property?",
    "What is the difference between MRTA and MLTA?",
    "What documents do I need to apply for a home loan?",
    "How does DSR affect my loan eligibility?",
    "What are the steps to buy my first house in Malaysia?",
    "What legal fees should I expect when buying property?",
    "How much fire insurance do I need for a bank loan?",
    "What is the SPA process?",
]

### Retriever Comparison

In [ ]:
def compare_retrievers(question, k=3):
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    
    kw = search_keyword(kw_index, question, k=k)
    vec = search_vector(embeddings, embedder, docs, question, k=k)
    hyb = search_hybrid(kw_index, embeddings, embedder, docs, question, k=k)
    
    for label, results in [("KEYWORD", kw), ("VECTOR", vec), ("HYBRID", hyb)]:
        print(f"\n--- {label} ---")
        for r in results:
            print(f"  [{r.get('score', 0):.4f}] {r['source']:20s}| {r['category']:15s} | {r['title'][:55]} ")

# Run first question as demo
compare_retrievers(test_questions[7])


Q: What is the SPA process?

--- KEYWORD ---
  [0.0000] iproperty           | buying_process  | The Process of Buying a House in Malaysia 2026 
  [0.0000] iproperty           | buying_process  | The Process of Buying a House in Malaysia 2026 
  [0.0000] iproperty           | buying_process  | The Process of Buying a House in Malaysia 2026 

--- VECTOR ---
  [0.4881] stashaway           | buying_process  | Complete Guide For First Time Home Buyer 
  [0.4551] iproperty           | buying_process  | The Process of Buying a House in Malaysia 2026 
  [0.4528] propcashflow        | legal_cost      | Stamp Duty (MOT) Malaysia 2026 

--- HYBRID ---
  [0.6041] iproperty           | buying_process  | The Process of Buying a House in Malaysia 2026 
  [0.5000] stashaway           | buying_process  | Complete Guide For First Time Home Buyer 
  [0.4272] iproperty           | buying_process  | The Process of Buying a House in Malaysia 2026 


### End-to-End RAG

In [ ]:
def test_rag(question, retriever="hybrid", rerank_results=False, prompt_template="default", k=3):
    result = ask(
        question=question,
        retriever=retriever,
        k=k,
        rerank_results=rerank_results,
        prompt_template=prompt_template,
    )
    print(f"Q: {question}")
    print(f"Retriever: {retriever} | Rerank: {rerank_results} | Template: {prompt_template}")
    print(f"Latency: {result['latency_ms']}ms")
    print(f"\nAnswer:\n{result['answer']}")
    print(f"\nSources:")
    for s in result['sources']:
        print(f"  - {s['title']} ({s['source']}) score={s['score']}")
    return result

# Test a few questions
for q in test_questions[:3]:
    test_rag(q)
    print("\n" + "-"*60 + "\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Q: How much stamp duty for a RM500k property?
Retriever: hybrid | Rerank: False | Template: default
Latency: 11337ms

Answer:
I’m sorry, but the provided context does not include the specific stamp‑duty rates or a calculation table needed to determine the duty on a RM 500,000 property. 【Source 1】【Source 2】【Source 3】

Sources:
  - Property Stamp Duty in Malaysia: How to Calculate (iproperty) score=0.75
  - Property Stamp Duty in Malaysia: How to Calculate (iproperty) score=0.4055
  - Property Stamp Duty in Malaysia: How to Calculate (iproperty) score=0.3448

------------------------------------------------------------

Q: What is the difference between MRTA and MLTA?
Retriever: hybrid | Rerank: False | Template: default
Latency: 1421ms

Answer:
MRTA (Mortgage Reducing Term Assurance) and MLTA (Mortgage Level Term Assurance) differ in how the coverage amount changes over the life of the loan.  

* **MRTA** – The sum‑insured drops as you repay the mortgage, so the protection amount reduce

### Prompt Template Comparison

In [ ]:
for template in ["default", "concise"]:
    print(f"\n{'='*60}")
    print(f"Template: {template}")
    print(f"{'='*60}")
    result = ask(
        question="What is the difference between MRTA and MLTA?",
        prompt_template=template,
    )
    print(result['answer'][:500])


Template: default
**MRTA (Mortgage Reducing Term Assurance)**  
- The sum‑insured falls as you repay the mortgage, so the coverage amount gets smaller over time.  
- Because the coverage declines, the premiums are generally lower than for MLTA.  
- It only pays out up to the amount of the sum‑insured; if your outstanding loan exceeds that amount, any shortfall must be paid by your family.  
- The policy is tied to the specific loan and cannot be transferred to another property.  

**MLTA (Mortgage Level Term Assu

Template: concise
**MRTA (Mortgage Reducing Term Assurance)** – The sum‑insured falls as you repay the loan, so the coverage amount gets smaller over time. Because the risk declines, premiums are generally lower. The policy is tied to the original loan and cannot be transferred to another property 【Source 1】【Source 2】  

**MLTA (Mortgage Level Term Assurance)** – The sum‑insured stays the same for the whole policy term, giving constant protection even as the loan balance dro

### All Test Questions

In [ ]:
all_results = []
for q in test_questions:
    result = ask(q)
    all_results.append({"question": q, **result})
    print(f"[{len(all_results)}/{len(test_questions)}] {q[:60]}... ({result['latency_ms']}ms)")

with open("results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("\nSaved results to test/results.json")

[1/8] How much stamp duty for a RM500k property?... (961ms)
[2/8] What is the difference between MRTA and MLTA?... (1189ms)
[3/8] What documents do I need to apply for a home loan?... (1538ms)
[4/8] How does DSR affect my loan eligibility?... (1728ms)
[5/8] What are the steps to buy my first house in Malaysia?... (1844ms)
[6/8] What legal fees should I expect when buying property?... (1766ms)
[7/8] How much fire insurance do I need for a bank loan?... (6921ms)
[8/8] What is the SPA process?... (7926ms)

Saved results to test/results.json


### Display Results

In [ ]:
import base64
from IPython.display import HTML, display
import pandas as pd
from typing import Any

def print_html(content: Any, title: str | None = None, is_image: bool = False):
    """
    Pretty-print inside a styled card.
    - If is_image=True and content is a string: treat as image path/URL and render <img>.
    - If content is a pandas DataFrame/Series: render as an HTML table.
    - Otherwise (strings/others): show as code/text in <pre><code>.
    """
    try:
        from html import escape as _escape
    except ImportError:
        _escape = lambda x: x

    def image_to_base64(image_path: str) -> str:
        with open(image_path, "rb") as img_file:
            return base64.b64encode(img_file.read()).decode("utf-8")

    # Render content
    if is_image and isinstance(content, str):
        b64 = image_to_base64(content)
        rendered = f'<img src="data:image/png;base64,{b64}" alt="Image" style="max-width:100%; height:auto; border-radius:8px;">'
    elif isinstance(content, pd.DataFrame):
        rendered = content.to_html(classes="pretty-table", index=False, border=0, escape=False)
    elif isinstance(content, pd.Series):
        rendered = content.to_frame().to_html(classes="pretty-table", border=0, escape=False)
    elif isinstance(content, str):
        rendered = f"<pre><code>{_escape(content)}</code></pre>"
    else:
        rendered = f"<pre><code>{_escape(str(content))}</code></pre>"

    css = """
    <style>
    .pretty-card{
      font-family: ui-sans-serif, system-ui;
      border: 2px solid transparent;
      border-radius: 14px;
      padding: 14px 16px;
      margin: 10px 0;
      background: linear-gradient(#fff, #fff) padding-box,
                  linear-gradient(135deg, #3b82f6, #9333ea) border-box;
      color: #111;
      box-shadow: 0 4px 12px rgba(0,0,0,.08);
    }
    .pretty-title{
      font-weight:700;
      margin-bottom:8px;
      font-size:14px;
      color:#111;
    }
    /* 🔒 Only affects INSIDE the card */
    .pretty-card pre, 
    .pretty-card code {
      background: #f3f4f6;
      color: #111;
      padding: 8px;
      border-radius: 8px;
      display: block;
      overflow-x: auto;
      font-size: 13px;
      white-space: pre-wrap;
    }
    .pretty-card img { max-width: 100%; height: auto; border-radius: 8px; }
    .pretty-card table.pretty-table {
      border-collapse: collapse;
      width: 100%;
      font-size: 13px;
      color: #111;
    }
    .pretty-card table.pretty-table th, 
    .pretty-card table.pretty-table td {
      border: 1px solid #e5e7eb;
      padding: 6px 8px;
      text-align: left;
    }
    .pretty-card table.pretty-table th { background: #f9fafb; font-weight: 600; }
    </style>
    """

    title_html = f'<div class="pretty-title">{title}</div>' if title else ""
    card = f'<div class="pretty-card">{title_html}{rendered}</div>'
    display(HTML(css + card))

    

In [ ]:
import json
with open("results.json") as f:
    results = json.load(f)


In [ ]:
import pandas as pd

pd.DataFrame(results)

,question,answer,sources,latency_ms
0,How much stamp duty for a RM500k property?,"I’m sorry, but the provided sources do not inc...",[{'id': '7bc86587-b76a-4895-8a58-0aaaa9a87929'...,1010
1,What is the difference between MRTA and MLTA?,**Difference between MRTA and MLTA**\n\n| Feat...,[{'id': '0fedbf15-b7fe-48f9-9363-111b9874e429'...,1422
2,What documents do I need to apply for a home l...,"To apply for a home loan in Malaysia, banks ty...",[{'id': '95ff7335-25d2-4727-9bea-14e532bbe97d'...,1131
3,How does DSR affect my loan eligibility?,Debt Service Ratio (DSR) is the key figure ban...,[{'id': 'fb0e843e-4a32-4584-9aad-bea8ad952029'...,1850
4,What are the steps to buy my first house in Ma...,Below is the typical sequence most first‑time ...,[{'id': '40298170-6d95-425f-a6a7-f453674f3bfe'...,1856
5,What legal fees should I expect when buying pr...,When you buy a property in Malaysia you’ll nee...,[{'id': '948b0eaf-b925-47e5-9fba-6851cd232949'...,8634
6,How much fire insurance do I need for a bank l...,The bank will only accept fire insurance that ...,[{'id': 'b16a5877-5c49-4c54-8256-685c8797f1ba'...,11222
7,What is the SPA process?,The SPA (Sales & Purchase Agreement) is the ke...,[{'id': '2bf96e20-40af-4a00-b363-528aa9f1f686'...,7468


In [ ]:
print_html(pd.DataFrame(results[0]))

question,answer,sources,latency_ms
How much stamp duty for a RM500k property?,"I’m sorry, but the provided sources do not include the specific stamp‑duty rates or a calculation table needed to determine the duty on a RM500,000 property. Therefore I can’t give you the exact amount based on the information given. \n\n*Source: Property Stamp Duty in Malaysia: How to Calculate (iproperty)*","{'id': '7bc86587-b76a-4895-8a58-0aaaa9a87929', 'title': 'Property Stamp Duty in Malaysia: How to Calculate', 'source': 'iproperty', 'url': 'https://www.iproperty.com.my/guides/stamp-duty-spa-legal-fees-owning-a-house-malaysia-24760', 'score': 0.75}",1010
How much stamp duty for a RM500k property?,"I’m sorry, but the provided sources do not include the specific stamp‑duty rates or a calculation table needed to determine the duty on a RM500,000 property. Therefore I can’t give you the exact amount based on the information given. \n\n*Source: Property Stamp Duty in Malaysia: How to Calculate (iproperty)*","{'id': '69b31808-03a8-4e9e-953c-3fd4e7965449', 'title': 'Property Stamp Duty in Malaysia: How to Calculate', 'source': 'iproperty', 'url': 'https://www.iproperty.com.my/guides/stamp-duty-spa-legal-fees-owning-a-house-malaysia-24760', 'score': 0.4873}",1010
How much stamp duty for a RM500k property?,"I’m sorry, but the provided sources do not include the specific stamp‑duty rates or a calculation table needed to determine the duty on a RM500,000 property. Therefore I can’t give you the exact amount based on the information given. \n\n*Source: Property Stamp Duty in Malaysia: How to Calculate (iproperty)*","{'id': 'bec65253-92ab-4fa6-91fc-6ce1b2d374ba', 'title': 'Property Stamp Duty in Malaysia: How to Calculate', 'source': 'iproperty', 'url': 'https://www.iproperty.com.my/guides/stamp-duty-spa-legal-fees-owning-a-house-malaysia-24760', 'score': 0.4411}",1010
How much stamp duty for a RM500k property?,"I’m sorry, but the provided sources do not include the specific stamp‑duty rates or a calculation table needed to determine the duty on a RM500,000 property. Therefore I can’t give you the exact amount based on the information given. \n\n*Source: Property Stamp Duty in Malaysia: How to Calculate (iproperty)*","{'id': 'a20453ef-1196-4198-82f1-9c64a7bba55b', 'title': 'The Process of Buying a House in Malaysia 2026', 'source': 'iproperty', 'url': 'https://www.iproperty.com.my/guides/documents-and-paperwork-buying-a-house-in-malaysia-71908', 'score': 0.2556}",1010
How much stamp duty for a RM500k property?,"I’m sorry, but the provided sources do not include the specific stamp‑duty rates or a calculation table needed to determine the duty on a RM500,000 property. Therefore I can’t give you the exact amount based on the information given. \n\n*Source: Property Stamp Duty in Malaysia: How to Calculate (iproperty)*","{'id': 'daa280a7-bb2a-4d7e-a4f5-338955244539', 'title': 'Property Stamp Duty in Malaysia: How to Calculate', 'source': 'iproperty', 'url': 'https://www.iproperty.com.my/guides/stamp-duty-spa-legal-fees-owning-a-house-malaysia-24760', 'score': 0.25}",1010
